In [ ]:
# Copyright (c) 2017-2019 Uber Technologies, Inc.
# SPDX-License-Identifier: Apache-2.0
import jax
import jax.numpy as jnp

import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS

numpyro.set_platform("cpu")  # change to "gpu" if desired


"""
This simple example is intended to demonstrate how to use an LKJ prior with
a multivariate distribution.

It generates entirely random, uncorrelated data, and then attempts to fit a correlation matrix
and vector of variances.
"""

numpyro.set_platform("cpu")  # change to "gpu" if desired
numpyro.set_host_device_count(1)

def model(y):
    N, d = y.shape

    # Vector of standard deviations
    theta = numpyro.sample(
        "theta",
        dist.HalfCauchy(jnp.ones(d))
    )

    # LKJ prior over correlation matrices
    concentration = 1.0
    L_omega = numpyro.sample(
        "L_omega",
        dist.LKJCholesky(dimension=d, concentration=concentration)
    )

    # covariance Cholesky
    L_Omega = jnp.diag(jnp.sqrt(theta)) @ L_omega

    mu = jnp.zeros(d)

    with numpyro.plate("observations", N):
        numpyro.sample(
            "obs",
            dist.MultivariateNormal(loc=mu, scale_tril=L_Omega),
            obs=y,
        )

rng_key = jax.random.PRNGKey(0)

N = 500
num_variables = 5

rng_key, subkey = jax.random.split(rng_key)
y = jax.random.normal(subkey, (N, num_variables))

nuts_kernel = NUTS(model)

mcmc = MCMC(
    nuts_kernel,
    num_warmup=100,
    num_samples=200,
    num_chains=1,
    progress_bar=True,
)

mcmc.run(rng_key, y)

mcmc.print_summary()